# Evaluation Analysis
Analyse experiment results — judge scores, latency, cost, and accuracy vs efficiency tradeoffs across candidate models.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv('../.env')
mongo = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017'))
db = mongo[os.getenv('MONGO_DB', 'data_flywheel')]
print('Connected')

## 1. Experiment Overview

In [ ]:
experiments = list(db.experiments.find({'status': 'completed'}))

if not experiments:
    print('No completed experiments yet — run: make run-flywheel')
else:
    rows = []
    for e in experiments:
        m = e.get('metrics', {})
        rows.append({
            'experiment_id': e['_id'][:8],
            'model_id': e['model_id'],
            'type': e['experiment_type'],
            'accuracy': m.get('accuracy'),
            'mean_score': m.get('mean_score'),
            'latency_p95_ms': m.get('latency_p95_ms'),
            'latency_p50_ms': m.get('latency_p50_ms'),
            'cost_per_1k': m.get('cost_per_1k_tokens'),
            'sample_count': m.get('sample_count'),
            'promoted': e.get('promoted', False),
        })

    df = pd.DataFrame(rows)
    print(f'Completed experiments: {len(df)}')
    display(df)

## 2. Judge Score Distribution by Model

In [ ]:
if experiments:
    models = df['model_id'].unique()
    colors = cm.tab10(np.linspace(0, 1, len(models)))

    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]

    for ax, model, color in zip(axes, models, colors):
        model_exps = [e for e in experiments if e['model_id'] == model]
        all_scores = []
        for e in model_exps:
            dist = e.get('metrics', {}).get('score_distribution', {})
            for score, count in dist.items():
                all_scores.extend([int(score)] * count)

        if all_scores:
            ax.hist(all_scores, bins=[0.5,1.5,2.5,3.5,4.5,5.5],
                    color=color, edgecolor='white', rwidth=0.8)
            ax.axvline(4, color='red', linestyle='--', alpha=0.7, label='Win threshold')
            win_rate = sum(1 for s in all_scores if s >= 4) / len(all_scores)
            ax.set_title(f'{model}\nWin rate: {win_rate:.1%}')
            ax.set_xlabel('Judge score (1-5)')
            ax.set_xticks([1, 2, 3, 4, 5])

    axes[0].set_ylabel('Count')
    plt.suptitle('Judge Score Distribution by Model', y=1.02)
    plt.tight_layout()
    plt.show()

## 3. Accuracy vs Latency vs Cost

In [ ]:
if experiments and len(df.dropna()) > 0:
    df_clean = df.dropna(subset=['accuracy', 'latency_p95_ms', 'cost_per_1k'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy vs Latency — bubble size = cost
    scatter = axes[0].scatter(
        df_clean['latency_p95_ms'],
        df_clean['accuracy'],
        s=df_clean['cost_per_1k'] * 20000,
        c=range(len(df_clean)),
        cmap='tab10',
        alpha=0.7,
        edgecolors='white',
    )
    axes[0].axhline(0.85, color='red', linestyle='--', alpha=0.7, label='Min accuracy (0.85)')
    axes[0].axvline(800, color='orange', linestyle='--', alpha=0.7, label='Max latency (800ms)')
    for _, row in df_clean.iterrows():
        axes[0].annotate(f"{row['model_id']}\n({row['type']})",
                         (row['latency_p95_ms'], row['accuracy']),
                         textcoords='offset points', xytext=(6, 4), fontsize=7)
    axes[0].set_xlabel('Latency p95 (ms)')
    axes[0].set_ylabel('Accuracy (win rate)')
    axes[0].set_title('Accuracy vs Latency\n(bubble size = cost per 1k tokens)')
    axes[0].legend(fontsize=8)

    # Accuracy vs Cost
    axes[1].scatter(
        df_clean['cost_per_1k'],
        df_clean['accuracy'],
        s=100,
        c=range(len(df_clean)),
        cmap='tab10',
        alpha=0.7,
        edgecolors='white',
    )
    axes[1].axhline(0.85, color='red', linestyle='--', alpha=0.7, label='Min accuracy')
    axes[1].axvline(0.02, color='orange', linestyle='--', alpha=0.7, label='Max cost ($0.02)')
    axes[1].set_xlabel('Cost per 1k tokens (USD)')
    axes[1].set_ylabel('Accuracy (win rate)')
    axes[1].set_title('Accuracy vs Cost')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 4. ICL vs LoRA SFT Comparison

In [ ]:
if experiments:
    df_icl = df[df['type'] == 'icl']
    df_lora = df[df['type'] == 'lora_sft']

    metrics = ['accuracy', 'latency_p95_ms', 'cost_per_1k']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, metric in zip(axes, metrics):
        data = [df_icl[metric].dropna().tolist(), df_lora[metric].dropna().tolist()]
        bp = ax.boxplot(data, labels=['ICL', 'LoRA SFT'], patch_artist=True)
        bp['boxes'][0].set_facecolor('steelblue')
        if len(bp['boxes']) > 1:
            bp['boxes'][1].set_facecolor('coral')
        ax.set_title(metric.replace('_', ' ').title())

    plt.suptitle('ICL vs LoRA SFT — Metric Comparison')
    plt.tight_layout()
    plt.show()